# Querying the Semantic Layer

Instead of writing SQL, SLayer queries describe **what** data you want: measures, dimensions, filters, and time granularity. SLayer generates and executes the SQL.

This notebook demonstrates common query patterns against the Jaffle Shop dataset, which has been auto-ingested with rollup joins across 7 tables.

See also: [Queries docs](../../concepts/queries.md) | [Formulas docs](../../concepts/formulas.md)

**Prerequisites:** `pip install motley-slayer[examples]`

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.query import SlayerQuery

engine, storage, models = ensure_jaffle_shop()

## Revenue by Store

The `orders` model has a join to `stores`, so we can group by `stores.name` directly — SLayer resolves the JOIN automatically:

In [ ]:
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}, {"formula": "order_total_sum"}],
    dimensions=[{"name": "stores.name"}],
    order=[{"column": {"name": "order_total_sum"}, "direction": "desc"}],
))

for row in result.data:
    store = row["orders.stores.name"]
    count = row["orders.count"]
    revenue = row["orders.order_total_sum"]
    print(f"  {store}: {count:,} orders, ${revenue:,.2f} revenue")

## Monthly Order Trends with Cumulative Sum

Time dimensions truncate a date/timestamp column to a given granularity. The `cumsum` formula function computes a running total over time:

In [ ]:
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    time_dimensions=[{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    fields=[
        {"formula": "count"},
        {"formula": "cumsum(count)", "name": "cumulative"},
    ],
    order=[{"column": {"name": "ordered_at"}, "direction": "asc"}],
))

print(f"{'Month':<12} {'Orders':>8} {'Cumulative':>12}")
print("-" * 34)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    count = row["orders.count"]
    cumulative = row["orders.cumulative"]
    print(f"{month:<12} {count:>8,} {cumulative:>12,}")

## Top Products by Quantity Sold

This query uses the `order_items` model, which has a join to `products`. The joined dimensions `products.name` and `products.type` are available automatically:

In [ ]:
result = engine.execute(query=SlayerQuery(
    source_model="order_items",
    fields=[{"formula": "count"}, {"formula": "quantity_sum"}],
    dimensions=[{"name": "products.name"}, {"name": "products.type"}],
    order=[{"column": {"name": "quantity_sum"}, "direction": "desc"}],
))

print(f"{'Product':<35} {'Type':<10} {'Line Items':>12} {'Qty Sold':>10}")
print("-" * 70)
for row in result.data:
    name = row["order_items.products.name"]
    ptype = row["order_items.products.type"]
    count = row["order_items.count"]
    qty = row["order_items.quantity_sum"]
    print(f"{name:<35} {ptype:<10} {count:>12,} {qty:>10,}")

## Month-over-Month Revenue Change

The `change()` and `change_pct()` formula functions compute the difference from the previous time period. They use self-join CTEs internally, so they handle gaps correctly and don't produce edge NULLs:

In [ ]:
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    time_dimensions=[{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    fields=[
        {"formula": "order_total_sum"},
        {"formula": "change(order_total_sum)", "name": "mom_change"},
        {"formula": "change_pct(order_total_sum)", "name": "mom_pct"},
    ],
    order=[{"column": {"name": "ordered_at"}, "direction": "asc"}],
    limit=12,
))

print(f"{'Month':<12} {'Revenue':>12} {'MoM Change':>12} {'MoM %':>8}")
print("-" * 48)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>11,.2f} {chg_str:>12} {pct_str:>8}")

## Filtering

Filters are condition strings that support SQL comparison operators (`=`, `<>`, `>`, `>=`, `<`, `<=`), `IN`, `IS NULL`, `LIKE`, and boolean logic (`AND`, `OR`). Filters on joined dimensions work the same way:

In [ ]:
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}, {"formula": "order_total_sum"}],
    dimensions=[{"name": "stores.name"}],
    filters=["stores.name in ('Brooklyn', 'Philadelphia')"],
    order=[{"column": {"name": "order_total_sum"}, "direction": "desc"}],
))

for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders.count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

## Arithmetic Fields

Field formulas can combine measures with arithmetic. Use `name` to give the computed column a short alias:

In [ ]:
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[
        {"formula": "count"},
        {"formula": "order_total_sum"},
        {"formula": "order_total_sum / count", "name": "avg_order_value"},
    ],
    dimensions=[{"name": "stores.name"}],
    order=[{"column": {"name": "avg_order_value"}, "direction": "desc"}],
))

print(f"{'Store':<20} {'Orders':>8} {'Revenue':>14} {'Avg Order':>12}")
print("-" * 58)
for row in result.data:
    print(f"{row['orders.stores.name']:<20} {row['orders.count']:>8,} ${row['orders.order_total_sum']:>13,.2f} ${row['orders.avg_order_value']:>11,.2f}")

## Summary

SLayer queries are declarative — you describe what data you want, not how to get it:

- **Dimensions** group the data (including joined dimensions like `stores.name`)
- **Fields** define computed columns: measures, arithmetic, and transform functions
- **Time dimensions** add time-based grouping with truncation granularity
- **Filters** restrict rows using familiar comparison syntax

JOINs, GROUP BY, and window functions are all generated automatically.

**Next:** See the [Joins](../joins/joins.ipynb) notebook for a deep dive into how joins work in SLayer.